## Import libraries

In [22]:
import pandas as pd
import numpy as np

# Show all columns
pd.set_option('display.max_columns', None)

## Load the dataset

In [23]:
merged_df = pd.read_csv(r'./data/merged.csv', low_memory=False)

Let's take a quick look at the dataset:

In [24]:
merged_df

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,CompetitionOpen
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,0,2015,7,31,82.0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2015,7,31,92.0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",2015,7,31,103.0
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,0,2015,7,31,70.0
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,0,2015,7,31,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844333,682,2,2013-01-01,3375,566,1,0,a,1,b,a,150.0,9.0,2006.0,0,0.0,0.0,0,2013,1,1,76.0
844334,733,2,2013-01-01,10765,2377,1,0,a,1,b,b,860.0,10.0,1999.0,0,0.0,0.0,0,2013,1,1,159.0
844335,769,2,2013-01-01,5035,1248,1,0,a,1,b,b,840.0,NaN,NaN,1,48.0,2012.0,"Jan,Apr,Jul,Oct",2013,1,1,0.0
844336,948,2,2013-01-01,4491,1039,1,0,a,1,b,b,1430.0,NaN,NaN,0,0.0,0.0,0,2013,1,1,0.0


## Train-Test split

In [25]:
# change the Dtype of Date column
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

In [26]:
# set the cutoff
cutoff = merged_df['Date'].max() - pd.Timedelta(weeks=6)

In [27]:
train = merged_df[merged_df['Date'] <= cutoff]
test = merged_df[merged_df['Date'] > cutoff]

In [28]:
X_train = train.drop('Sales', axis=1)
X_test = test.drop('Sales', axis=1)
y_train = train['Sales']
y_test = test['Sales']

Let's decide Input and Traget columns:

In [29]:
input_cols = ['DayOfWeek', 'Customers', 'Open',
              'Promo', 'StateHoliday', 'SchoolHoliday',
              'StoreType', 'Assortment', 'CompetitionDistance',
              'CompetitionOpen','Promo2', 'Promo2SinceWeek',
              'Promo2SinceYear', 'Year', 'Month', 'Day']

target_cols = 'Sales'

Let's separate `categorical` and `numerical` columns from input columns for `Encoding` and `Scaling` respectively,
note that we can treat binary categorical columns(0/1) as numeric columns *(Because with only two values, the “distance between values” problem disappears)*:

In [30]:
cat_cols = ['DayOfWeek', 'StoreType', 'Assortment', 'StateHoliday']

num_cols = ['Customers', 'Open', 'Promo', 'SchoolHoliday',
            'CompetitionDistance', 'CompetitionOpen', 'Promo2', 'Promo2SinceWeek',
            'Promo2SinceYear', 'Year', 'Month', 'Day']

## Preprocessing

#### Encoding categorical columns

In [31]:
from sklearn.preprocessing import OneHotEncoder

In [32]:
encoder = OneHotEncoder(sparse_output=False)

In [33]:
X_train_encoded = encoder.fit_transform(X_train[cat_cols])
X_test_encoded = encoder.transform(X_test[cat_cols])

#### Scaling numerical columns

In [34]:
from sklearn.preprocessing import StandardScaler

In [35]:
scaler = StandardScaler()

In [36]:
X_train_scaled = scaler.fit_transform(X_train[num_cols])
X_test_scaled = scaler.transform(X_test[num_cols])

#### Combine Encoded and Scaled columns

In [37]:
X_train_final = np.hstack([X_train_encoded, X_train_scaled])
X_test_final = np.hstack([X_test_encoded, X_test_scaled])

Let's check their shape before we go further:

In [38]:
X_train_final.shape, X_test_final.shape

((804056, 30), (40282, 30))

## Baseline Random Forest

In [39]:
from sklearn.ensemble import RandomForestRegressor

In [40]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

Let's train this baseline model:

In [41]:
rf.fit(X_train_final, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

Let's predict using this trained baseline model:

In [42]:
rf_preds = rf.predict(X_test_final)

Let's evaluate our baseline model using `MAE`, and `R-Squared` from `sklearn`:

In [43]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [44]:
rf_mae = mean_absolute_error(y_test, rf_preds)
print('Baseline Random Forest MAE:', rf_mae)

Baseline Random Forest MAE: 392.44207555483837


Our baseline **Random forest** model is off by about **€392** (in either direction).

In [45]:
y_test_mean = y_test.mean()
print('y_test_mean:', y_test_mean)

y_test_mean: 6978.638051735266


Our basline **Random Forest**'s MAE is about `5.6%` of the **average daily sales**.\
To put it simply: if average daily sales were 100, the model would typically predict somewhere around 95 to 105.

In [46]:
rf_r2 = r2_score(y_test, rf_preds)
print('Baseline Random Forest R-Sqauared:', rf_r2)

Baseline Random Forest R-Sqauared: 0.9615149375036403


Our baseline model is able to explain 96% of the variance in daily sales.

**Next:** Let's check feature importnace our baseline **Random Forest** but first let's get all the features:

In [ ]:
# Features from encoding
encoded_features = encoder.get_feature_names_out()

In [ ]:
# Features from scaling
scaled_features = scaler.get_feature_names_out()

In [ ]:
# concatenate features from encoding and scaling
all_features = np.concatenate([encoded_features, scaled_features])

In [82]:
pd.DataFrame({
    'Feature': all_features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

,Feature,Importance
18,Customers,0.742715
22,CompetitionDistance,0.069057
10,StoreType_d,0.043926
20,Promo,0.032906
12,Assortment_b,0.019645
23,CompetitionOpen,0.018412
25,Promo2SinceWeek,0.014962
8,StoreType_b,0.009965
26,Promo2SinceYear,0.008581
28,Month,0.007021


So according to our basline random forest, the most important feature is **`Customers`**, meaning a store with\
most customers will have the most sales on everyday basis.

## Baseline XGBoost

In [47]:
from xgboost import XGBRegressor

In [48]:
xgb = XGBRegressor(random_state=42, n_jobs=-1)

Let's train our baseline XGBoost model:

In [49]:
xgb.fit(X_train_final, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

Let's predict using this trained baseline model:

In [50]:
xgb_preds = xgb.predict(X_test_final)

Now, Let's evaluate our basline model using the same matrics as before:

In [51]:
xgb_mae = mean_absolute_error(y_test, xgb_preds)
print("Baseline XGBoost MAE:", xgb_mae)

Baseline XGBoost MAE: 501.51959228515625


Our baseline **XGBoost** model is off by about **€501.5**, which is worse than our baseline **Random forest** model (€392).

In [52]:
y_test_mean = y_test.mean()
print('y_test_mean:', y_test_mean)

y_test_mean: 6978.638051735266


Our basline **XGBoost**'s MAE is about `7%` of the **average daily sales**.

In [53]:
xgb_r2 = r2_score(y_test, xgb_preds)
print("Baseline XGBoost R-Squared:", xgb_r2)

Baseline XGBoost R-Squared: 0.9441004395484924


Baseline **XGBoost** is able to explain `94.41%` of vriance in daily sales, though baseline **Random Forest** was `96%`.

**Next:** Let's check feature importnace our baseline **XGBoost**:

In [83]:
pd.DataFrame({
    'Feature': all_features,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

,Feature,Importance
12,Assortment_b,0.272462
18,Customers,0.238685
8,StoreType_b,0.133995
10,StoreType_d,0.112155
20,Promo,0.042391
13,Assortment_c,0.026552
0,DayOfWeek_1,0.021242
24,Promo2,0.015961
22,CompetitionDistance,0.015887
25,Promo2SinceWeek,0.011690


Interesting, our baseline **XGBoost** is prioritizing `Assortment_b` over `Customers`(2nd most importnat).